# Generate stochastic model output for umap
Use Austin's parameters for first pass.  
Ensure each saved cell cycle trajectory through high dimensional space is truncated from peak to peak of cdc20 (mitosis)  


In [1]:
import os
import sys
sys.path.insert(0, os.getcwd() + "/cellular_cycle") #add a pathway to the model 
sys.path.insert(0, os.getcwd() + "/fit") #add a pathway to the fit functions 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tempfile import TemporaryDirectory
import time


##Importing the model, models differ on their initial conditions, one being a concentration and the other a protein count
from Austin_proliferation_model_concentration_5 import model
model_concentration=model

from Austin_proliferation_model_population_5 import model
model_population=model

##Import the simulator: for ODE and stochastic simulation :
from pysb.simulator import BngSimulator
from pysb.simulator import ScipyOdeSimulator

from runAKTERK_2 import runAKTERK #Simulate a population of cells from independants biochemical models
from runStochCell import runStochcell

from retrieve_cellState_dw import retrieve_cellState #
from call_cell_events import call_events #
from SMs_G1s_IMTs_extraction import IMTs

import fracprolif as fp
from rbf_mmd import nmmd2_rbf



#change the model parameters of the model if needed
omega = 200
new_param = pd.read_csv(os.getcwd() + "/../umap_window/parameters.csv") 
del new_param['Stress']
#for key in new_param.keys():
#    if model_population.parameters[key].value != new_param[key][0]:
#        model_population.parameters[key].value = new_param[key]
#    if model_concentration.parameters[key].value != new_param[key][0]:
#        model_concentration.parameters[key].value = new_param[key]

# Directory containing cell fate data
from pathlib import Path



In [2]:
SAVEFILES = True
OVERWRITE = True

DATADIR = Path('../Data/Exp_Intermitotic_Times/')

SAVEDIR = Path(os.path.expanduser("~/ECM_mod_output"))
if not os.path.exists(SAVEDIR) and SAVEFILES:
    os.mkdir(SAVEDIR)
    
# DATADIR = "../../Data/Exp_Cell_Counts"

# rcParams.update({'figure.autolayout': True})



In [3]:
%%time
initial_time = time.time()

conditions = ['0.2 kPa DMSO', '0.2 kPa ERL', '4 kPa DMSO', '4 kPa ERL', '55 kPa DMSO', '55 kPa ERL']
for condition_exp in conditions:
    if 'DMSO' in condition_exp:  
        print('DMSO')
        if '0.2 kPa' in condition_exp :
            print('0.6 kPa')
            model_population.parameters['ERK'].value = 1.0 * omega ###
            model_population.parameters['FAK'].value = 0.4 * omega
            model_population.parameters['AKT'].value = 0.6 * omega ###
        elif '4 kPa' in condition_exp :
            model_population.parameters['ERK'].value = 1.0 * omega
            model_population.parameters['FAK'].value = 0.6 * omega
            model_population.parameters['AKT'].value = 1.0 * omega
        else : #55 kPa
            print('55 kPa')
            model_population.parameters['ERK'].value = 1 * omega
            model_population.parameters['FAK'].value = 1 * omega
            model_population.parameters['AKT'].value = 1 * omega
    else : #ERL treated
        print('ERL')
        if '0.2 kPa' in condition_exp :
            print('0.6 kPa')
            model_population.parameters['ERK'].value = 0.2 * omega
            model_population.parameters['FAK'].value = 0.4 * omega
            model_population.parameters['AKT'].value = 0.3 * omega
        elif '4 kPa' in condition_exp :
            print('4 kPa')
            model_population.parameters['ERK'].value = 0.2 * omega
            model_population.parameters['FAK'].value = 0.6 * omega
            model_population.parameters['AKT'].value = 0.6 * omega
        else : #55 kPa
            print('55 kPa')
            model_population.parameters['ERK'].value = 0.2 * omega
            model_population.parameters['FAK'].value = 1.0 * omega
            model_population.parameters['AKT'].value = 0.6 * omega

    #how long is the experiment
    run_time = 120 

    # how long are we looking at new cells
    observed_cells_time = 120 
    
    # size of initial population
    nber_of_starting_cells = 5 
    
    condition = condition_exp
    
    index="_" #to be removed...
    print(f"Time from start is {np.round(time.time()-initial_time, 3)} seconds")
    (cells, cell_count) = runAKTERK(nber_of_starting_cells, run_time, observed_cells_time, model_concentration, model_population, index, condition);


DMSO
0.6 kPa
Time from start is 0.0 seconds
ERL
0.6 kPa
Time from start is 44.767 seconds
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/darren/miniconda3/envs/ECM_model/lib/python3.8/site-packages/IPython/core/magics/execution.py", line 1316, in time
    exec(code, glob, local_ns)
  File "<timed exec>", line 52, in <module>
  File "/Users/darren/git-repos/ECM_cell_cycle_model/Python/cellular_cycle/runAKTERK_2.py", line 44, in runAKTERK
  File "/Users/darren/git-repos/ECM_cell_cycle_model/Python/cellular_cycle/runStochCell.py", line 148, in runStochcell
    return variable_range, check, tmp_score
UnboundLocalError: local variable 'variable_range' referenced before assignment

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/darren/miniconda3/envs/ECM_model/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 1993, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
  File "/Users/darren/miniconda3/envs/ECM_model/lib/python3.8/site-packages/IPython/core/ultratb.py", line 1